# Chapter 11 &mdash; Applying the CFL Pumping Lemma, and Non-Closure Under Intersection

**Concept 19 of the Chapter 11 decomposition:** *Applying the CFL Pumping Lemma, and Non-Closure Under Intersection*

$L_{ww}$ is not context-free via $0^N1^N0^N1^N$; and two CFLs intersect to a non-CFL.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Applying-CFL-Pumping/Concept-Applying-CFL-Pumping.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]



import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The template, applied to $L_{ww} = \{ww : w\in\{0,1\}^*\}$:

1. choose $s = 0^N1^N0^N1^N$, which is $ww$ with $w = 0^N1^N$;
2. $|vxy|\le N$ confines $vxy$ to **at most two adjacent blocks**;
3. pumping therefore changes one or two blocks and **not** their mirror images;
4. the two halves no longer match, so $uv^2xy^2z \notin L_{ww}$.

The choice in step 1 is what makes step 2 easy &mdash; that is the art.

The same machinery settles **non-closure under intersection**:
$\{a^nb^nc^m\} \cap \{a^mb^nc^n\} = \{a^nb^nc^n\}$, and the CFL pumping lemma shows
the result is not context-free.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The two languages, as predicates

In [ ]:
def in_ww(s):
    return len(s) % 2 == 0 and s[:len(s)//2] == s[len(s)//2:]

def in_anbncn(s):
    i = len(s) - len(s.lstrip('a')); rest = s[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    return s == 'a'*i + 'b'*j + 'c'*k and i == j == k

### The five-way split machinery

In [ ]:
def splits5(w, N):
    out, n = [], len(w)
    for i in range(n+1):
        for j in range(i, n+1):
            for k in range(j, n+1):
                for l in range(k, n+1):
                    u, v, x, y, z = w[:i], w[i:j], w[j:k], w[k:l], w[l:]
                    if not (v or y): continue
                    if len(v + x + y) > N: continue
                    out.append((u, v, x, y, z))
    return out

def pump(sp, i):
    u, v, x, y, z = sp
    return u + v*i + x + y*i + z

<!-- nav-strip -->

---

&larr;&nbsp;[Ch11&nbsp;18.&nbsp;The Pumping Lemma for Context-Free Languages](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-CFL-Pumping-Lemma/Concept-CFL-Pumping-Lemma.ipynb) &nbsp;&middot;&nbsp; [**Chapter 11** index](https://github.com/ganeshutah/Jove/blob/master/Chapter11/README.md) &nbsp;&middot;&nbsp; [Ch11&nbsp;20.&nbsp;The Complement of a Non-CFL Can Be a CFL: $\overline{L_{ww}}$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Complement-Of-Lww/Concept-Complement-Of-Lww.ipynb)&nbsp;&rarr;

---

## 3. Tests

**$L_{ww}$ is not context-free.** Every admissible split breaks.

In [ ]:
for N in [2, 3]:
    s = '0'*N + '1'*N + '0'*N + '1'*N
    assert in_ww(s)
    sp = splits5(s, N)
    broken = [x for x in sp if any(not in_ww(pump(x, i)) for i in range(3))]
    print("N=%d : s=%s, %d admissible splits, %d broken"
          % (N, s, len(sp), len(broken)))
    assert len(broken) == len(sp)

**Why the choice of $s$ matters.** A careless $s$ leaves survivors.

In [ ]:
N = 3
bad_s = '0'*(2*N) + '0'*(2*N)              # still in L_ww, but all one block
assert in_ww(bad_s)
sp = splits5(bad_s, N)
surv = [x for x in sp if all(in_ww(pump(x, i)) for i in range(3))]
print("s = %r : %d of %d splits SURVIVE" % (bad_s, len(surv), len(sp)))
assert surv
print("\n0^2N 0^2N is all one block, so pumping inside it can keep the halves equal.")
print("0^N 1^N 0^N 1^N forces vxy into at most two ADJACENT blocks -- that is the art.")

**$\{a^nb^nc^n\}$ is not context-free** by the same template.

In [ ]:
for N in [2, 3]:
    s = 'a'*N + 'b'*N + 'c'*N
    assert in_anbncn(s)
    sp = splits5(s, N)
    broken = [x for x in sp if any(not in_anbncn(pump(x, i)) for i in range(3))]
    print("N=%d : s=%s, %d splits, %d broken" % (N, s, len(sp), len(broken)))
    assert len(broken) == len(sp)
print("\n|vxy| <= N means vxy touches at most two of the three blocks,")
print("so pumping cannot keep all three counts equal.")

**Non-closure under intersection**, assembled.

In [ ]:
G1 = mkg({'S': ["XC"], 'X': ["", "aXb"], 'C': ["", "cC"]})
G2 = mkg({'S': ["AY"], 'A': ["", "aA"], 'Y': ["", "bYc"]})
L1, L2 = set(language(G1, 6)), set(language(G2, 6))
inter = sorted(L1 & L2, key=lambda s: (len(s), s))
print("L1 (a^n b^n c^m) is CF, L2 (a^m b^n c^n) is CF")
print("intersection :", inter)
assert all(in_anbncn(w) for w in inter)
print("\n= a^n b^n c^n, which we just proved is NOT context-free.")
print("Therefore the CFLs are not closed under intersection.")

And, by DeMorgan, not under complement either.

In [ ]:
print("closed under union  : yes")
print("closed under complement : if it were, intersection would follow")
print("intersection : NO")
print("=> complement : NO")

## 4. Exercises


1. Prove $\{a^ib^jc^k : i<j<k\}$ is not context-free.
2. Which step of the template is the creative one?
3. Is $L_{ww^R}$ context-free? Why does the same choice of $s$ not break it?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter11/Concept-Applying-CFL-Pumping')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')